# Plot TXx time series

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import xarray as xr

from utils import berkeley, era5, hadex3, utils

In [ ]:
CLIMATOLOGY_PERIOD = slice("1961", "1990")

## Load ERA5 data

In [ ]:
era5_txx_orig = era5.load_post(variable="txx").t2m
# era5_tnn_orig = era5.load_post(variable="tnn").t2m

era5_txx_orig = era5_txx_orig.set_index(time="year").rename(time="year")
era5_land_mask = era5.load_landmask()

In [ ]:
era5_txx_land = utils.land_mean(era5_txx_orig, era5_land_mask > 0.5)
era5_txx_land = utils.calc_anomaly(era5_txx_land, CLIMATOLOGY_PERIOD)

## Load Berkeley Earth data

In [ ]:
berkeley_txx_99_90_land_1961_1990 = berkeley.read_post(
    "txx", "99_90_land_1961_1990"
).txx

## Load HadEX3

> for HadEX3 the same methodology is used as in the [IPCC AR6 WGI Chapter 11, Figure 11.2](https://www.ipcc.ch/report/ar6/wg1/figures/chapter-11/figure-11-2/), see also [IPCC-WG1/Chapter-11/code/Figure_11.2_obs_ts_plots.ipynb](https://github.com/IPCC-WG1/Chapter-11/blob/main/code/Figure_11.2_obs_ts_plots.ipynb). 

In [ ]:
# time period where data must be non-NaN
hadex_time_valid = slice("1961", None)

hadex_landmask = hadex3.HadEx3.read_landmask()


def hadex_calc_ave(dta, lat=None, minimum_valid=0.9):

    dta = dta.sel(time=hadex_time_valid)

    dta = hadex3.valid_for_globmean(dta, minimum_valid=minimum_valid)

    dta = dta if lat is None else dta.sel(lat=lat)

    dta = utils.land_mean(dta, hadex_landmask)

    return utils.calc_anomaly(dta, CLIMATOLOGY_PERIOD, dim="time")


HADEX_txx_map_orig = hadex3.HadEx3.read_file("TXx")
HADEX_txx = hadex_calc_ave(HADEX_txx_map_orig)

# for consistency
HADEX_txx = HADEX_txx.rename(time="year")

## Load offset

Computed offset of from 1961-1990 to 1850-1900 based on Berkeley Earth, see [BerkeleyEarth_process_data.ipynb](BerkeleyEarth_process_data.ipynb) for details.

In [ ]:
berkeley_txx_offset = berkeley.read_post("txx", "offset_99_90_land_1961_1990").offset

## Figure

Only plot TXx for now, due to the differences in TNn which need to be understood first

In [ ]:
f, ax = plt.subplots(1, 1, layout="constrained", sharex=True, sharey=True)

f.set_size_inches(17 / 2.54, 9 / 2.54)

# plot ERA5
d = era5_txx_land + berkeley_txx_offset
d.plot(ax=ax, label="ERA5", color="#33a02c")

# plot Berkeley Earth
d = berkeley_txx_99_90_land_1961_1990 + berkeley_txx_offset
d.plot(ax=ax, label="Berkeley Earth", color="#1f78b4")

# plot HadEX3
d = HADEX_txx + berkeley_txx_offset
d.plot.line(ax=ax, label="HadEX3", color="#ff7f00")

# style plot

ax.legend()

ax.set_xlabel("")

ax.set_ylabel("Anomaly w.r.t. 1850–1900 (°C)")
ax.set_title("Land average annual maximum temperature anomaly")

ax.axhline(0, color="0.1", lw=0.5)

sns.despine(f)


plt.savefig(utils.fig_dir / "txx_ts_1961_1990_offset.png", dpi=300)
plt.savefig(utils.fig_dir / "txx_ts_1961_1990_offset.pdf", dpi=300)

## Write data

In [ ]:
def ts_to_pandas(era5, berkeley, hadex):

    # set names
    era5.name = "ERA5"
    berkeley.name = "Berkeley"
    hadex.name = "HadEX3"

    txx = xr.merge(
        [era5_txx_land, berkeley_txx_99_90_land_1961_1990, HADEX_txx], join="outer"
    )

    txx = txx.to_pandas()

    return txx


df_txx = ts_to_pandas(era5_txx_land, berkeley_txx_99_90_land_1961_1990, HADEX_txx)

In [ ]:
fN = "../data/txx_timeseries.csv"


HEADER = f"""\
Annual maximum land-mean temperatures (TXx) for
- ERA5
- Berkeley
- HadEX3

- units: °C
- land mean, no gap filling
- anomalies with respect to 1961-1990
- to obtain values relative to 1850-1900 please add offset
- offset: {berkeley_txx_offset.item():0.2f}°C

NOTE: using this data requires attributing the climate
      indicator project and the data providers. See
      https://github.com/ClimateIndicator/igcc_t_extremes#data

"""


with open(fN, "w") as fid:

    fid.writelines(HEADER)
    df_txx.to_csv(fid, na_rep="-", float_format="%.2f")

In [ ]:
df_txx_offset = df_txx + berkeley_txx_offset.item()

df_txx_offset.round(2)

### Difference between the last two years

In [ ]:
diff = df_txx_offset.iloc[-1] - df_txx_offset.iloc[-2]

diff.round(2)

## Temporal means

Decadal means

In [ ]:
periods = (
    utils.Decade(2000, remove=False),
    utils.Decade(2004),
    utils.Decade(2005),
    utils.Decade(2006, remove=False),  # ten years before the current decade
    utils.Decade(2007),
    utils.Decade(2008),
    utils.Decade(2009, remove=False),  # last decade to include HadEX3
    utils.Decade(2010),
    utils.Decade(2011),
    utils.Decade(2012),
    utils.Decade(2013),
    utils.Decade(2014, remove=False),
    utils.Decade(2015, remove=False),
    utils.Decade(2016, remove=False),
)

periods

In [ ]:
out = dict()
for period in periods:

    mn_hadex = df_txx["HadEX3"].loc[period.slice].mean(skipna=False)
    mn_berkeley = df_txx["Berkeley"].loc[period.slice].mean(skipna=False)
    mn_era5 = df_txx["ERA5"].loc[period.slice].mean(skipna=False)

    out[period.key] = [mn_hadex, mn_berkeley, mn_era5]

df_decadal = pd.DataFrame.from_dict(out).T
df_decadal += berkeley_txx_offset.item()

columns = ("HadEX3", "Berkeley Earth", "ERA5")

df_decadal.columns = columns

# df_decadal = df_decadal.round(2)

df_decadal.index.name = "Period"

df_decadal

In [ ]:
fN = "../data/txx_decadal_means_wrt_1850-1900.csv"

df_decadal_out = df_decadal.copy(True)

# merge the multiindex columns for a nicer csv
mi = df_decadal_out.columns
df_decadal_out.columns = mi

df_decadal_out.to_csv(fN, na_rep="   -", float_format="%.2f")

In [ ]:
# output all periods for comparison with last year

print(df_decadal.to_csv(sep="\t", na_rep="-", float_format="%0.2f"))

In [ ]:
# remove some periods
# NOTE: paste as plain text (not HTML formatted) to a spreadsheet

periods_to_remove = [period.key for period in periods if period.remove]

print(
    df_decadal.drop(index=periods_to_remove).to_csv(
        sep="\t", na_rep="-", float_format="%0.2f"
    )
)

In [ ]:
diff = df_decadal.loc["2015–2024"] - df_decadal.loc["2005–2014"]

diff.round(2)

In [ ]:
diff = df_decadal.loc["2016–2025"] - df_decadal.loc["2006–2015"]

diff.round(2)

In [ ]:
df_decadal